In [1]:
import pandas as pd
import numpy as np

# Выносим станции-призраки в отдельный датасет

Датасет all_years и stations используются для постройки дашборда в DataLens

In [3]:
after_2020 = pd.read_parquet('..//data//after_2020.parquet')
before_2020 = pd.read_parquet('..//data//before_2020.parquet')
stations = pd.read_csv('..//data//stations.csv')

In [4]:
after_2020 = after_2020.reset_index(drop=True)
after_2020 = after_2020.dropna(subset=['start_station_name', 'end_station_name'])

In [5]:
THRESHHOLD_FOR_STATIONS = 1000

all_years = pd.concat([before_2020, after_2020])

ssc = all_years.start_station_name.value_counts()
ssc = ssc > THRESHHOLD_FOR_STATIONS
ssc = ssc.rename('start_to_left')

esc = all_years.end_station_name.value_counts()
esc = esc > THRESHHOLD_FOR_STATIONS
esc = esc.rename('end_to_left')

all_years = all_years.merge(ssc, on='start_station_name', how='left')
all_years = all_years.merge(esc, on='end_station_name', how='left')

In [6]:
all_years = all_years[(all_years['start_to_left']) & (all_years['end_to_left'])]

In [7]:
ghost_stations = esc[~esc].index
ghost_stations = stations[stations['station_name'].isin(ghost_stations)]

In [8]:
after_2020.to_parquet('..//data//after_2020_income_included.parquet')
before_2020.to_parquet('..//data//before_2020.parquet')

# Считаем все доходы и расходы

Мы выбрали распределение подписок как 35% на 65% опираясь [на это исследование](https://medium.com/%40aswinpushkar11/exploratory-data-analysis-of-divvy-bike-sharing-a5ce4498e723)

Число подписчиков Divvy оценить тяжелее, так как нет публичных отчетов. Оценка составлена на [заявлении Чикагской администрации](https://www.chicago.gov/city/en/depts/cdot/provdrs/bike/news/2023/april/divvy-for-the-entire-city--divvy-service-hits-all-50-wards.html?)

>Last year (2022) Divvy had nearly 550,000 unique riders and reached over 43,000 members.

In [10]:
after_2020 = pd.read_parquet('..//data//after_2020_income_included.parquet')

In [11]:
def calculate_revenues(df, business_params):
    results = []

    for year in range(2020, 2025):
        rides = df[df['started_at'].dt.year == year]
        ride_revenue = rides['income'].sum()

        subs = business_params['annual_pass'].get(year, {})
        if 'price' in subs:
            sub_revenue = subs['count'] * subs['price']
        else:
            sub_revenue = (subs['count'] * 0.35 * subs['price_basic'] +
                          subs['count'] * 0.65 * subs['price_premium'])

        total_revenue = ride_revenue + sub_revenue

        results.append({
            'year': year,
            'total_revenue': total_revenue
        })

    return pd.DataFrame(results)

In [12]:
def calculate_expenses(df, business_params):
    expenses = []

    for year in range(2020, 2025):
        year_data = df[df['started_at'].dt.year == year]
        unique_stations = pd.concat([
            year_data['start_station_name'],
            year_data['end_station_name']
        ]).nunique()

        station_costs = unique_stations * business_params['station_rent'][year] * 12

        salary_costs = (business_params['employees_count'][year] *
                       business_params['employee_salary'][year] * 12)

        total_expenses = station_costs + salary_costs

        expenses.append({
            'year': year,
            'total_expenses': total_expenses
        })

    return pd.DataFrame(expenses)

In [13]:
df = after_2020.copy()
df['started_at'] = pd.to_datetime(df['started_at'])

price_count_ect = {
    'station_rent': {
        2020: 300,
        2021: 325,
        2022: 350,
        2023: 375,
        2024: 400,
    }, # средняя стоимость аренды и обсуживания одной станции и всех ее велосипедов в месяц
    'employee_salary': {
        2020: 3000,
        2021: 3100,
        2022: 3200,
        2023: 3300,
        2024: 3400,
    }, # средняя зарплата сотрудника в месяц (средняя с 2020)
    'employees_count': {
        2020: 150,
        2021: 150,
        2022: 155,
        2023: 155,
        2024: 155,
    },
    'annual_pass': {
        2020: {'count': 35000, 'price': 99},
        2021: {'count': 40000, 'price': 108},
        2022: {'count': 45000, 'price': 119},
        2023: {'count': 50000, 'price_basic': 130.9, 'price_premium': 199},
        2024: {'count': 55000, 'price_basic': 143.9, 'price_premium': 199},
    }
}

In [14]:
revenues = calculate_revenues(df, price_count_ect)
expenses = calculate_expenses(df, price_count_ect)

In [15]:
final_report = revenues.merge(expenses, on='year')
final_report['profit'] = final_report['total_revenue'] - final_report['total_expenses']

final_report.to_csv('..//data//summary.csv', index=False)

## Юнит экономика для каждого вида байка

In [16]:
final_report = pd.read_csv('..//data//summary.csv')
bikes_amount = pd.read_csv('..//data//bikes_amount.tsv', sep='\t')

final_report = final_report.merge(bikes_amount, on='year')

In [17]:
final_report['profit_per_bike'] = final_report['profit'] / (final_report['classic_bikes'] + final_report['electric_bikes'])

In [18]:
after_2020_temp = after_2020.copy()
after_2020_temp['year'] = after_2020_temp['started_at'].dt.year

after_2020_temp['rideable_type'] = after_2020_temp['rideable_type'].map({'classic_bike': 'classic_bike',
                                               'docked_bike': 'classic_bike',
                                                'electric_scooter': 'electric_bike',
                                                'electric_bike': 'electric_bike',
                                                'Scooters': 'electric_bike'
                                               })

In [19]:
profits = pd.DataFrame(columns=['year', 'bike_type', 'profit'])

profits = profits.astype({
    'year': 'int16',
    'bike_type': 'object',
    'profit': 'int64'
})

for year in range(2020, 2025):
    for bike_type in ['electric_bike', 'classic_bike']:
        p_type = after_2020_temp[after_2020_temp['year'] == year]['rideable_type'].value_counts(normalize=True).get(bike_type, 0)

        P = final_report[final_report['year'] == year]['profit_per_bike'].values[0]
        bike_share = final_report[final_report['year'] == year][bike_type + 's'] / (final_report['classic_bikes'] + 
                                                                              final_report['electric_bikes'])
        bike_share = bike_share.dropna().values[0]
        
        profit = round(p_type * P / bike_share, 2)
        profits.loc[len(profits)] = [year, bike_type, profit]

In [20]:
final_report.to_csv('..//data//final_report.csv', index=False)
profits.to_csv('..//data//profits.csv', index=False)